# Credit Default Risk Analysis
**Brian Owino Abayo** | Python, Pandas, Scikit-learn

## Objective
Explore a public credit-default dataset, identify patterns associated with next-month default, and evaluate a logistic-regression baseline. This is a portfolio exercise, not a production lending model.

**Dataset note:** UCI Default of Credit Card Clients (Taiwan). It is not Kenyan borrower data.

## 1. Load libraries and data

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, RocCurveDisplay, PrecisionRecallDisplay
sys.path.insert(0, str(Path.cwd().resolve().parent / 'src'))
from analysis import load_data, build_pipeline

data = load_data()
print('Shape:', data.shape)
display(data.head())


## 2. Initial data checks

In [ ]:
display(data.info())
display(data.isna().sum().sort_values(ascending=False).head(15))
print('Duplicate rows:', data.duplicated().sum())
print('Target distribution:')
display(data['default_next_month'].value_counts(normalize=True).rename('share'))


## 3. Exploratory data analysis

In [ ]:
target_counts = data['default_next_month'].value_counts().rename(index={0:'No default', 1:'Default'})
ax = target_counts.plot(kind='bar', title='Next-month default distribution', ylabel='Number of clients')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Compare numeric features by target. Focus first on repayment status variables.
pay_cols = [c for c in data.columns if 'PAY_' in str(c).upper()]
if pay_cols:
    display(data.groupby('default_next_month')[pay_cols].mean().T)


## 4. Train/test split and baseline model

In [ ]:
X = data.drop(columns='default_next_month')
y = data['default_next_month']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
model = build_pipeline(X_train)
model.fit(X_train, y_train)
prob = model.predict_proba(X_test)[:, 1]
pred = (prob >= 0.50).astype(int)
print(classification_report(y_test, pred, zero_division=0))
print('ROC-AUC:', round(roc_auc_score(y_test, prob), 4))


## 5. Evaluation

In [ ]:
cm = confusion_matrix(y_test, pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted no default','Predicted default'],
            yticklabels=['Actual no default','Actual default'])
plt.title('Confusion matrix — threshold 0.50')
plt.tight_layout()
plt.show()

RocCurveDisplay.from_predictions(y_test, prob)
plt.title('ROC curve — logistic regression baseline')
plt.tight_layout()
plt.show()

PrecisionRecallDisplay.from_predictions(y_test, prob)
plt.title('Precision–recall curve')
plt.tight_layout()
plt.show()


## 6. Interpretation and limitations

Complete this section after running the notebook. Discuss:
- Which variables show noticeable differences across default groups?
- How does recall compare with precision at the 0.50 threshold?
- What do false negatives and false positives mean in a lending context?
- Why might a Taiwan credit-card dataset not transfer to Kenyan microfinance?
- What additional data governance, fairness, calibration and validation would be needed?

Do not claim that associations are causal. Do not present this model as suitable for real lending decisions.